In [17]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [18]:
!pip install ftfy nltk gensim pandas numpy scikit-learn
print('Installation complete')

Installation complete


In [19]:
!pip install vaderSentiment langdetect --quiet

In [20]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('✅ NLTK data downloaded')

✅ NLTK data downloaded


In [21]:
BASE = '/content/drive/MyDrive/Text Miners/Actual Work'
print('Connected:', BASE)

Connected: /content/drive/MyDrive/Text Miners/Actual Work


In [22]:
import sys
sys.path.append(f'{BASE}/pre-processing')
import preprocess

print('✅ preprocess.py imported')

✅ preprocess.py imported


In [23]:
import pandas as pd
from gensim.models import Word2Vec
import pickle

# Load the labelled reviews (this should have sentiment labels)
df = pd.read_csv(f'{BASE}/data/labelled_reviews.csv')
print(f'Loaded {len(df)} labelled reviews')
print(df.head())

# Load the Word2Vec model you created earlier
w2v_model = Word2Vec.load(f'{BASE}/features/word2vec.model')
print(f'\n✅ Word2Vec model loaded')
print(f'Vocabulary size: {len(w2v_model.wv)}')

Loaded 380505 labelled reviews
   listing_id                                           comments  \
0       27934  We stayed in the apartment for a week and we e...   
1       27934  My girlfriend and I recently stayed in Nuttee'...   
2       27934  I stayed for one month at the condo and was re...   
3       27934  Nuttee was a great host! I really enjoyed her ...   
4       27934  Nuttee was an amazing host. She and her daught...   

   compound_score     label  
0          0.9729  positive  
1          0.9875  positive  
2          0.9913  positive  
3          0.9925  positive  
4          0.9775  positive  

✅ Word2Vec model loaded
Vocabulary size: 28321


In [24]:
# Debug: Check what columns we have
print('All columns in dataframe:')
print(df.columns.tolist())
print('\nDataframe shape:', df.shape)
print('\nFirst 3 rows:')
print(df.head(3))

All columns in dataframe:
['listing_id', 'comments', 'compound_score', 'label']

Dataframe shape: (380505, 4)

First 3 rows:
   listing_id                                           comments  \
0       27934  We stayed in the apartment for a week and we e...   
1       27934  My girlfriend and I recently stayed in Nuttee'...   
2       27934  I stayed for one month at the condo and was re...   

   compound_score     label  
0          0.9729  positive  
1          0.9875  positive  
2          0.9913  positive  


In [25]:
# Preprocess the reviews to create 'cleaned_text' and 'tokens' columns
df = preprocess.preprocess_dataframe(df, text_col='comments')

print('✅ Reviews preprocessed')
print(f'Columns now: {df.columns.tolist()}')
print(df[['comments', 'tokens']].head(2))

✅ Reviews preprocessed
Columns now: ['listing_id', 'comments', 'compound_score', 'label', 'cleaned_text', 'tokens']
                                            comments  \
0  We stayed in the apartment for a week and we e...   
1  My girlfriend and I recently stayed in Nuttee'...   

                                              tokens  
0  [stayed, apartment, week, enjoyed, very, much,...  
1  [girlfriend, recently, stayed, nuttee, condo, ...  


In [26]:
import pandas as pd
from gensim.models import Word2Vec

# Load the labelled reviews
df = pd.read_csv(f'{BASE}/data/labelled_reviews.csv')
print(f'Loaded {len(df)} labelled reviews')
print('Columns:', df.columns.tolist())

# Preprocess to create 'tokens' column
df = preprocess.preprocess_dataframe(df, text_col='comments')
print('\n✅ Preprocessing complete')
print(df[['comments', 'cleaned_text', 'tokens']].head(2))

# Load Word2Vec model
w2v_model = Word2Vec.load(f'{BASE}/features/word2vec.model')
print(f'\n✅ Word2Vec model loaded')
print(f'Vocabulary size: {len(w2v_model.wv)}')

Loaded 380505 labelled reviews
Columns: ['listing_id', 'comments', 'compound_score', 'label']

✅ Preprocessing complete
                                            comments  \
0  We stayed in the apartment for a week and we e...   
1  My girlfriend and I recently stayed in Nuttee'...   

                                        cleaned_text  \
0  stayed apartment week enjoyed very much nuttee...   
1  girlfriend recently stayed nuttee condo month ...   

                                              tokens  
0  [stayed, apartment, week, enjoyed, very, much,...  
1  [girlfriend, recently, stayed, nuttee, condo, ...  

✅ Word2Vec model loaded
Vocabulary size: 28321


In [27]:
import numpy as np

# Build vocabulary
vocab = {word: idx+1 for idx, word in enumerate(w2v_model.wv.index_to_key)}
vocab['<PAD>'] = 0
vocab_size = len(vocab)

print(f'Vocabulary size: {vocab_size}')

# Encoding function
def encode_review(tokens, vocab, max_length=100):
    ids = [vocab.get(token, 0) for token in tokens]

    if len(ids) > max_length:
        ids = ids[:max_length]

    if len(ids) < max_length:
        ids = ids + [0] * (max_length - len(ids))

    return ids

# Encode all reviews
MAX_LENGTH = 100
df['encoded_review'] = df['tokens'].apply(
    lambda tokens: encode_review(tokens, vocab, MAX_LENGTH)
)

print(f'\n✅ Encoded {len(df)} reviews')

Vocabulary size: 28322

✅ Encoded 380505 reviews


In [32]:
import torch
import torch.nn as nn

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes, embedding_matrix=None):
        super(TextCNN, self).__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))

        # Three parallel Conv1D filters
        self.conv1 = nn.Conv1d(in_channels=embedding_dim, out_channels=100, kernel_size=2)
        self.conv2 = nn.Conv1d(in_channels=embedding_dim, out_channels=100, kernel_size=3)
        self.conv3 = nn.Conv1d(in_channels=embedding_dim, out_channels=100, kernel_size=4)

        # Dropout
        self.dropout = nn.Dropout(0.5)

        # Linear classifier
        self.fc = nn.Linear(300, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        x1 = torch.relu(self.conv1(x))
        x1 = torch.max_pool1d(x1, x1.size(2))

        x2 = torch.relu(self.conv2(x))
        x2 = torch.max_pool1d(x2, x2.size(2))

        x3 = torch.relu(self.conv3(x))
        x3 = torch.max_pool1d(x3, x3.size(2))

        x = torch.cat([x1, x2, x3], dim=1)
        x = x.squeeze(2)

        x = self.dropout(x)
        x = self.fc(x)

        return x

# Create embedding matrix from Word2Vec
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in vocab.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print(f'✅ Embedding matrix shape: {embedding_matrix.shape}')

# Initialize model
num_classes = df['label'].nunique()
model = TextCNN(vocab_size, embedding_dim, num_classes, embedding_matrix)

print(f'✅ TextCNN model created')
print(f'Number of classes: {num_classes}')
print(model)

✅ Embedding matrix shape: (28322, 100)
✅ TextCNN model created
Number of classes: 3
TextCNN(
  (embedding): Embedding(28322, 100, padding_idx=0)
  (conv1): Conv1d(100, 100, kernel_size=(2,), stride=(1,))
  (conv2): Conv1d(100, 100, kernel_size=(3,), stride=(1,))
  (conv3): Conv1d(100, 100, kernel_size=(4,), stride=(1,))
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=300, out_features=3, bias=True)
)


In [33]:
# Convert string labels to numeric
label_mapping = {label: idx for idx, label in enumerate(df['label'].unique())}
df['label_numeric'] = df['label'].map(label_mapping)

print('Label mapping:', label_mapping)
print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nNumeric labels created:')
print(df[['label', 'label_numeric']].head())

Label mapping: {'positive': 0, 'neutral': 1, 'negative': 2}

Label distribution:
label
positive    359559
neutral      14784
negative      6162
Name: count, dtype: int64

Numeric labels created:
      label  label_numeric
0  positive              0
1  positive              0
2  positive              0
3  positive              0
4  positive              0


In [35]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# Prepare data
X = np.array(df['encoded_review'].tolist())
y = df['label_numeric'].values  # ✅ Fixed

# Convert to PyTorch tensors
X_tensor = torch.LongTensor(X)
y_tensor = torch.LongTensor(y)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f'Training on: {device}')
print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}')

print('✅ Training complete')

Training on: cpu
Training samples: 304404
Test samples: 76101
Epoch 1/10, Loss: 0.0954
Epoch 2/10, Loss: 0.0895
Epoch 3/10, Loss: 0.0859
Epoch 4/10, Loss: 0.0820
Epoch 5/10, Loss: 0.0773
Epoch 6/10, Loss: 0.0758
Epoch 7/10, Loss: 0.0712
Epoch 8/10, Loss: 0.0695
Epoch 9/10, Loss: 0.0653
Epoch 10/10, Loss: 0.0631
✅ Training complete


In [36]:
from sklearn.metrics import classification_report, f1_score

# Evaluate on test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.numpy())

# Classification report
print('\n📊 Classification Report:')
print(classification_report(all_labels, all_preds))

# F1 scores
f1_textcnn = f1_score(all_labels, all_preds, average='weighted')
print(f'\n✅ TextCNN F1 Score: {f1_textcnn:.4f}')


📊 Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99     71912
           1       0.69      0.58      0.63      2909
           2       0.70      0.51      0.59      1280

    accuracy                           0.97     76101
   macro avg       0.79      0.69      0.74     76101
weighted avg       0.96      0.97      0.97     76101


✅ TextCNN F1 Score: 0.9659
